# Bank Statement Anomaly Detection — PDF → Image → Embedding → Score

This notebook classifies bank-statement documents as **normal** or **anomaly** by comparing each
one against a set of **known-genuine** statements. It is fully self-contained: everything needed to
run it on a fresh machine — requirements, folder layout, and the complete pipeline code — is inside
this file. Nothing else from the project is required.

### The method in one paragraph
We turn each **page** into a 768-number "fingerprint" (an *embedding*) using a frozen vision model
(Microsoft **DiT**). We then model the fingerprints of genuine statement **pages** as a single cloud
of points (a Gaussian) and measure how far each new page falls from the centre of that cloud — its
**Mahalanobis distance**. Pages that fall far outside the genuine cloud are flagged as anomalies, and
a document is flagged if **any** of its pages is. This is a *one-class* approach (the PaDiM paradigm):
we only ever learn what **genuine** looks like — no forged/fake examples are needed, and the alarm
threshold is derived purely from how much genuine pages naturally vary among themselves.

### Every page is checked — not just the first
A fraudster can leave page 1 genuine and forge the pages behind it. So the pipeline rasterizes and
scores **every page** of every document, reports a score per page, and pinpoints *which* page is
suspicious — a faked page 7 behind a real page 1 is caught and named.

### The four pipeline stages (run per page)
| Stage | Input → Output | Section |
|-------|----------------|---------|
| 1. Rasterize | PDF/image bytes → one RGB image **per page** | Run · Stage 1 |
| 2. Embed | each page image → 768-dim vector (frozen DiT, on CPU) | Run · Stage 2 |
| 3. Fit "normal" model | all genuine **pages** → Gaussian + alarm threshold | Run · Stage 3 |
| 4. Score | each page → distance → per-page verdict; a document is **anomaly if any page is** | Run · Stage 4 |

### How to run it (any machine)
1. Install the **requirements** (next section) — all pure `pip`, **nothing else to install**.
2. Put your documents in the **folder structure** shown two sections down.
3. From the menu choose **Run → Run All Cells**. The first run downloads the DiT model weights
   (~330 MB) once, then caches them. On a laptop CPU, expect roughly 5–10 seconds **per page**.

## 1 · Requirements

**Python:** 3.10 or newer.

**Everything installs with `pip` — there is no separate system software to install.** (PDF reading uses
`pypdfium2`, which bundles Google's PDFium engine inside the pip package, so — unlike the older
`pdf2image`/Poppler route — nothing extra is needed on the machine.)

| Package | Purpose |
|---------|---------|
| `torch` | runs the DiT neural network |
| `transformers` | loads the DiT model + its image pre-processor from Hugging Face |
| `pypdfium2` | renders a PDF page to an image (self-contained; **no Poppler / no system install**) |
| `pillow` | image decoding/handling |
| `numpy` | the maths (Gaussian, Mahalanobis distance) |
| `matplotlib` | the charts |

Run the cell below once (safe to re-run). That's the whole setup.

> **Offline note:** the first run downloads the DiT model weights from Hugging Face (~330 MB, cached
> afterwards). If the target machine has no internet, copy the sender's `~/.cache/huggingface` folder
> to the same path on the target machine, or set the `HF_HOME` environment variable to a folder that
> already contains the weights.

A ready-to-save `requirements.txt` (same as the install cell below):
```
torch>=2.2
transformers>=4.40
pypdfium2>=4
pillow>=10
numpy>=1.26
matplotlib>=3.8
```

In [ ]:
# Install everything. Pure pip — no system software needed. Safe to re-run (pip skips what's present).
%pip install -q "torch>=2.2" "transformers>=4.40" "pypdfium2>=4" "pillow>=10" "numpy>=1.26" "matplotlib>=3.8"

In [ ]:
# Environment check — confirms every requirement is present. There is NO system dependency to set up:
# PDF reading is handled by pypdfium2, which ships its rendering engine inside the pip package.
import importlib.util, sys

print("Python:", sys.version.split()[0], "(need >= 3.10)\n")
print("Python packages:")
all_ok = True
for mod, pretty in [("torch","torch"),("transformers","transformers"),("pypdfium2","pypdfium2"),
                    ("PIL","pillow"),("numpy","numpy"),("matplotlib","matplotlib")]:
    ok = importlib.util.find_spec(mod) is not None
    all_ok = all_ok and ok
    print(f"  [{'OK ' if ok else 'MISSING'}] {pretty}")

print("\n" + ("All requirements present — no system software needed."
              if all_ok else
              "Something is MISSING — run the install cell above, then re-run this cell."))

## 2 · Folder structure

Keep this notebook at the **root of a working folder**, with `reference/` and `data/` beside it. The
`result/` folder is created for you.

```
your_working_folder/
├── layout_based_anomaly_detection.ipynb   ← this notebook
│
├── reference/
│   └── bank_statement/        ← KNOWN-GENUINE statements. These define "normal".
│       ├── BCA.pdf                Use at least 3 (the model needs 3+); 12–20 is much better.
│       ├── BNI.pdf                One clean, genuine statement per bank/layout you expect.
│       └── …
│
├── data/
│   └── bank_statement/        ← the documents you want to CHECK (classify).
│       ├── statement_001.pdf      These are scored against the genuine set above.
│       └── …
│
└── result/                    ← created automatically; holds the timestamped results CSV.
    └── bank_statement/
        └── 20260717_101500_results.csv
```

**What each part is for**

| Path | Role |
|------|------|
| `reference/bank_statement/` | The ground truth of "genuine". The model is built **only** from these, so they must all be real, clean statements. Adding more here makes the alarm threshold more reliable. |
| `data/bank_statement/` | The documents under question. Each is labelled `normal` or `anomaly`. |
| `result/bank_statement/` | Output. One `…_results.csv` per run (timestamped, so runs never overwrite each other). |
| accepted file types | `.pdf`, `.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`. PDFs use the **first page only**. Sub-folders and other file types are ignored. |

To analyse a **different** document type later (e.g. payslips), make `reference/<type>/` and
`data/<type>/` folders and change `DOC_TYPE` in the configuration cell below — nothing else changes.

**No ground-truth reference? Use *unsupervised mode*.** If you just have a pile of documents and no
separate set of known-genuine ones, put them all in `reference/<type>/`, leave `data/<type>/` empty,
and set `UNSUPERVISED = True` in the config cell. The notebook then scores every document by how
unusual it is *versus the rest of the pile* (leave-one-out) and flags the most unusual `PERCENTILE`%
— no ground truth and no manual threshold needed.

> The next cell **auto-detects** the working folder by looking for a `reference/` folder in the
> notebook's directory and its parents, so the notebook works whether you keep it beside `reference/`
> and `data/` or inside a `notebooks/` subfolder. To force a specific location, hard-code
> `BASE_DIR` in that cell.

In [ ]:
from pathlib import Path
import torch

# ── Configuration you can edit ──────────────────────────────────────────────
DOC_TYPE   = "bank_statement"       # the subfolder name under reference/ and data/
MODEL      = "microsoft/dit-base"   # frozen embedding model (DiT, 768-dim)
PERCENTILE = 95.0                   # alarm threshold = this percentile of genuine scores

# DEVICE: auto-detects a GPU (CUDA) and falls back to CPU if none is available/usable, so this
# notebook runs unchanged on a GPU or CPU-only machine. To force one or the other, hard-code
# DEVICE = "cuda" or DEVICE = "cpu" instead of the line below.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# UNSUPERVISED = True  ->  you have NO ground-truth reference set. Put your whole pile of documents in
# reference/<type>/ (leave data/<type>/ empty). Every document is then scored by how unusual it is
# versus the REST of the same set (leave-one-out), and the most unusual PERCENTILE% are flagged.
UNSUPERVISED = False
# ────────────────────────────────────────────────────────────────────────────

# BASE_DIR is the folder that holds reference/ and data/. We auto-detect it by searching the
# current folder and its parents for a "reference/" folder, so this works whether the notebook
# sits next to reference/ and data/ or inside a subfolder. To override, just hard-code
# BASE_DIR to an absolute path.
def _find_base(start: Path, doc_type: str) -> Path:
    for cand in (start, *start.parents):
        if (cand / "reference" / doc_type).is_dir() or (cand / "reference").is_dir():
            return cand
    return start
BASE_DIR = _find_base(Path.cwd(), DOC_TYPE)

REFERENCE_DIR = BASE_DIR / "reference" / DOC_TYPE
DATA_DIR      = BASE_DIR / "data" / DOC_TYPE
RESULT_DIR    = BASE_DIR / "result" / DOC_TYPE
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DEVICE   : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "  (no GPU detected — set up CUDA drivers/torch to use one)"))
print("BASE_DIR :", BASE_DIR)
for label, d in [("reference", REFERENCE_DIR), ("data", DATA_DIR)]:
    status = "OK" if d.is_dir() else "MISSING — create it and add documents"
    print(f"  {label:10}: {d}   [{status}]")

## 3 · Pipeline code (self-contained)

The next four cells contain the entire pipeline. They mirror the production service so results match
it exactly, but are inlined here so this notebook needs nothing else. Read the comment at the top of
each cell to see what it does; you don't need to change anything.

- **3.1 `document_to_pages`** — PDF/image bytes → one RGB image **per page** *(mirrors `pipeline/embedding.py`)*
- **3.2 `DiTEmbedder`** — page image → 768-dim vector *(mirrors `pipeline/embedding.py`)*
- **3.3 `NormalModel`** — fit the genuine-page cloud, derive the threshold, score pages *(mirrors `anomaly.py`)*
- **3.4 plotting helpers** — the per-page distribution and the document×page map

In [ ]:
# 3.1 — Rasterize: raw document bytes -> a LIST of RGB images, ONE PER PAGE.
#       Every page of a PDF is rendered (via pypdfium2 — no system dependency), so fraud on any
#       page is visible. A plain image (jpg/png/tif) is a single-page document.
import io
from PIL import Image

PDF_RENDER_DPI = 200   # resolution used to rasterize PDF pages (200 is a good default)

def document_to_pages(data: bytes) -> list:
    """Return every page of the document as a list of RGB PIL images."""
    if not data:
        raise ValueError("empty document")
    if data[:5] == b"%PDF-":
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(data)                 # reads raw bytes; engine is bundled in the wheel
        try:
            if len(pdf) == 0:
                raise ValueError("PDF contains no pages")
            images = []
            for i in range(len(pdf)):                  # EVERY page, not just the first
                page = pdf[i]
                images.append(page.render(scale=PDF_RENDER_DPI / 72).to_pil().convert("RGB"))
                page.close()
            return images
        finally:
            pdf.close()
    try:
        return [Image.open(io.BytesIO(data)).convert("RGB")]
    except Exception as exc:
        raise ValueError(f"unrecognized document format: {exc}") from exc

In [ ]:
# 3.2 — Embed: an image -> a fixed 768-dim vector, using a FROZEN DiT model (no training).
#       microsoft/dit-base ships without a trained pooler, so we read out the MEAN of the patch
#       tokens (readout="mean"). The vector is the only thing scoring ever sees.
class DiTEmbedder:
    def __init__(self, model_id: str = "microsoft/dit-base", readout: str = "mean", device: str = "cpu"):
        import torch
        from transformers import AutoImageProcessor, AutoModel
        self.model_id, self.readout, self._device, self._torch = model_id, readout, device, torch
        self._processor = AutoImageProcessor.from_pretrained(model_id)
        # use_safetensors=True: this pinned torch (2.2.2, needed for CPU compatibility - see
        # requirements.txt) is older than what recent transformers demands for the legacy
        # torch.load pickle format (CVE-2025-32434). Safetensors loading is unaffected by that
        # restriction, so we force it explicitly rather than upgrading torch (which would crash
        # on this CPU - see requirements.txt for why).
        self._model = AutoModel.from_pretrained(model_id, use_safetensors=True).to(device).eval()
        self.dim = int(self._model.config.hidden_size)

    def embed(self, image) -> tuple:
        torch = self._torch
        inputs = self._processor(images=image, return_tensors="pt").to(self._device)
        with torch.no_grad():
            hidden = self._model(**inputs).last_hidden_state   # (1, 1 + n_patches, dim)
        vec = hidden[0, 0] if self.readout == "cls" else hidden[0, 1:].mean(dim=0)
        return tuple(vec.detach().cpu().numpy().astype("float32").tolist())

In [ ]:
# 3.3 — The "normal" model: fit a Gaussian to the genuine vectors and score new ones by
#       Mahalanobis distance. Pure numpy. Mirrors anomaly.py exactly.
import numpy as np
from dataclasses import dataclass

def _shrunk_covariance(centered):
    # Ledoit-Wolf shrinkage toward a scaled identity, so the covariance is invertible even with
    # far fewer samples (e.g. 15) than dimensions (768).
    n, d = centered.shape
    sample = (centered.T @ centered) / n
    mean_var = np.trace(sample) / d
    denom = ((sample - mean_var * np.eye(d)) ** 2).sum() / d
    numer = sum(((np.outer(centered[k], centered[k]) - sample) ** 2).sum() for k in range(n))
    beta = min(numer / (n**2 * d), denom)
    shrink = beta / denom if denom > 0 else 1.0
    return shrink * mean_var * np.eye(d) + (1 - shrink) * sample, float(shrink)

def _fit_gaussian(X):
    mu = X.mean(axis=0)
    cov, shrink = _shrunk_covariance(X - mu)
    return mu, np.linalg.inv(cov), shrink

def _mahalanobis(x, mu, cov_inv):
    v = x - mu
    return float(np.sqrt(max(v @ cov_inv @ v, 0.0)))

@dataclass
class NormalModel:
    mu: np.ndarray
    cov_inv: np.ndarray
    threshold: float
    shrinkage: float
    ref_scores: np.ndarray     # leave-one-out Mahalanobis scores of the reference set
    pca_basis: np.ndarray      # (2, d) top-2 principal directions of the reference set
    ref_pca: np.ndarray        # (n, 2) reference set projected to 2-D
    explained_var: tuple

    @classmethod
    def fit(cls, X, threshold_percentile: float = 95.0) -> "NormalModel":
        X = np.asarray(X, dtype=float)
        n = len(X)
        if n < 3:
            raise ValueError(f"need at least 3 reference vectors to fit a model, got {n}")
        mu, cov_inv, shrink = _fit_gaussian(X)
        # Honest threshold: score each reference vector against the OTHERS (leave-one-out),
        # so the threshold isn't optimistic.
        loo = np.array([_mahalanobis(X[i], *_fit_gaussian(np.delete(X, i, axis=0))[:2]) for i in range(n)])
        threshold = float(np.percentile(loo, threshold_percentile))
        centered = X - mu
        _, singular, vt = np.linalg.svd(centered, full_matrices=False)
        variance = (singular**2) / (singular**2).sum()
        basis = vt[:2]
        return cls(mu, cov_inv, threshold, shrink, loo, basis, centered @ basis.T,
                   (float(variance[0]), float(variance[1])))

    def score(self, x) -> float:
        return _mahalanobis(np.asarray(x, dtype=float), self.mu, self.cov_inv)

    def is_anomaly(self, x) -> bool:
        return self.score(x) > self.threshold

    def project(self, x) -> np.ndarray:
        return (np.asarray(x, dtype=float) - self.mu) @ self.pca_basis.T

In [ ]:
# 3.4 — Plotting helpers. Every chart labels its X and Y axes.
# (They save to a PNG and display it, so they work with any matplotlib backend.)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TEAL, RED, GREY, INK = "#0c7a71", "#b3352b", "#7b8783", "#16201f"
SUPPORTED = {".pdf", ".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def list_documents(folder: Path):
    if not folder.is_dir():
        raise SystemExit(f"folder not found: {folder}")
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)

def _short_name(name: str, limit: int = 30) -> str:
    if len(name) <= limit:
        return name
    head = (limit - 1) * 3 // 5
    return f"{name[:head]}…{name[-(limit - 1 - head):]}"

def _stem(name: str) -> str:
    return name.rsplit(".", 1)[0]

def plot_page_distribution(model, index, scores, verdicts, path, title_label="", unsupervised=False):
    """One dot per page. Supervised: genuine reference pages (teal) vs data pages (teal/red).
    Unsupervised: a single population — each page's LEAVE-ONE-OUT score — coloured by verdict.
    The threshold line is drawn; only anomalous pages are labelled so the chart stays readable."""
    fig, ax = plt.subplots(figsize=(10, 5.0))
    hi = max([model.threshold, float(model.ref_scores.max()), *scores]) * 1.12
    ref_y, data_y = 0.12, 0.28
    ax.axvspan(model.threshold, hi, color=RED, alpha=0.06, lw=0)
    ax.axvline(model.threshold, color=GREY, ls="--", lw=1.3)
    ax.text(model.threshold + hi * 0.008, 0.99, f"threshold {model.threshold:.0f}", color=GREY,
            fontsize=8, ha="left", va="top", transform=ax.get_xaxis_transform())
    rng = np.random.default_rng(0)
    if not unsupervised:   # supervised: show the genuine reference cloud as its own lane
        ax.scatter(model.ref_scores, ref_y + rng.uniform(-0.025, 0.025, len(model.ref_scores)),
                   s=22, c=TEAL, edgecolors="white", linewidths=0.6, zorder=3)
    lane_y = (ref_y + data_y) / 2 if unsupervised else data_y
    dy = lane_y + rng.uniform(-0.03, 0.03, len(scores))
    cols = [RED if v == "anomaly" else TEAL for v in verdicts]
    ax.scatter(scores, dy, s=42, c=cols, edgecolors="white", linewidths=0.9, zorder=4)
    levels = [0.44, 0.55, 0.66, 0.77, 0.88, 0.99]
    anoms = sorted((i for i, v in enumerate(verdicts) if v == "anomaly"), key=lambda i: scores[i])
    for rank, i in enumerate(anoms):
        name, pg, npg = index[i]
        y = levels[rank % len(levels)]
        tag = _short_name(_stem(name), 16) + (f" p{pg}" if npg > 1 else "")
        ax.plot([scores[i], scores[i]], [lane_y + 0.02, y - 0.03], color=RED, lw=0.5, alpha=0.4, zorder=2)
        ax.annotate(tag, (scores[i], y), ha="center", va="center", fontsize=6.5, color=RED,
                    fontweight="bold", zorder=5)
    ax.set_ylim(0, 1.06); ax.set_yticks([]); ax.set_xlim(0, hi)
    ax.spines[["left", "right", "top"]].set_visible(False)
    if unsupervised:
        ax.set_xlabel("Anomaly score — leave-one-out Mahalanobis distance (higher = more unusual vs the rest)", fontsize=9)
        ax.set_ylabel("individual documents (spread vertically to avoid overlap)", fontsize=9)
        ax.set_title("Anomaly scores — unsupervised (each document vs the rest)" + (f" — {title_label}" if title_label else ""),
                     fontsize=12, fontweight="bold", loc="left")
        handles = [
            Line2D([], [], marker="o", ls="", mfc=TEAL, mec="white", label="document — normal"),
            Line2D([], [], marker="o", ls="", mfc=RED, mec="white", label="document — anomaly (past threshold)"),
            Line2D([], [], color=GREY, ls="--", label="threshold"),
        ]
    else:
        ax.set_xlabel("Per-page anomaly score — Mahalanobis distance from genuine pages (higher = more unusual)", fontsize=9)
        ax.set_ylabel("individual pages (spread vertically to avoid overlap)", fontsize=9)
        ax.set_title("Per-page anomaly scores" + (f" — {title_label}" if title_label else ""),
                     fontsize=12, fontweight="bold", loc="left")
        handles = [
            Line2D([], [], marker="o", ls="", mfc=TEAL, mec="white", label="genuine reference page"),
            Line2D([], [], marker="o", ls="", mfc=TEAL, mec="white", label="data page — normal"),
            Line2D([], [], marker="o", ls="", mfc=RED, mec="white", label="data page — anomaly"),
            Line2D([], [], color=GREY, ls="--", label="threshold"),
        ]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.26),
              ncol=len(handles), fontsize=8, frameon=False)
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_score_histogram(scores, threshold, path, title_label=""):
    """The 'distribution' view for unsupervised runs: a histogram of anomaly scores. Bars past the
    threshold are red; the dashed line is the threshold (the 95th percentile of these same scores)."""
    scores = np.asarray(scores, dtype=float)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    bins = min(45, max(12, int(np.sqrt(len(scores)) * 1.5)))
    counts, edges, patches = ax.hist(scores, bins=bins, color=TEAL, edgecolor="white", linewidth=0.6)
    for patch, left in zip(patches, edges[:-1]):
        if left >= threshold:
            patch.set_facecolor(RED)
    ax.axvline(threshold, color=GREY, ls="--", lw=1.4)
    ax.text(threshold, ax.get_ylim()[1] * 0.98, f"  threshold {threshold:.1f}", color=GREY, fontsize=8, va="top")
    n_anom = int((scores > threshold).sum())
    ax.set_xlabel("Anomaly score — leave-one-out Mahalanobis distance (higher = more unusual)", fontsize=9)
    ax.set_ylabel("number of documents", fontsize=9)
    ax.set_title(f"Distribution of anomaly scores — {n_anom} of {len(scores)} above threshold"
                 + (f" · {title_label}" if title_label else ""), fontsize=12, fontweight="bold", loc="left")
    ax.spines[["right", "top"]].set_visible(False)
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_page_map(index, scores, verdicts, threshold, path, title_label=""):
    """A grid: one row per document, one square per page, coloured by anomaly score. Anomalous pages
    get a bold red outline and their score printed in the cell — so you can point at 'page 7 is fake'."""
    docs = []
    for name, pg, npg in index:
        if name not in docs:
            docs.append(name)
    ymap = {d: i for i, d in enumerate(docs)}
    maxpage = max(pg for _, pg, _ in index)
    fig, ax = plt.subplots(figsize=(max(6.0, 1.8 + maxpage * 0.55), max(3.0, 1.3 + len(docs) * 0.46)))
    xs = [pg for _, pg, _ in index]
    ys = [ymap[name] for name, _, _ in index]
    edge = [RED if v == "anomaly" else "#c9d2d0" for v in verdicts]
    lw = [2.4 if v == "anomaly" else 0.6 for v in verdicts]
    sc = ax.scatter(xs, ys, c=scores, cmap="YlOrRd", vmin=0, vmax=max(scores), s=320, marker="s",
                    edgecolors=edge, linewidths=lw, zorder=3)
    for (name, pg, npg), s, v in zip(index, scores, verdicts):
        if v == "anomaly":
            ax.text(pg, ymap[name], f"{s:.0f}", ha="center", va="center",
                    fontsize=6, fontweight="bold", color="black", zorder=4)
    ax.set_yticks(range(len(docs))); ax.set_yticklabels([_short_name(d, 26) for d in docs], fontsize=8)
    ax.set_xticks(range(1, maxpage + 1))
    ax.set_xlim(0.4, maxpage + 0.6); ax.set_ylim(-0.6, len(docs) - 0.4)
    ax.invert_yaxis()
    ax.set_xlabel("Page number", fontsize=9); ax.set_ylabel("Document", fontsize=9)
    ax.set_title("Per-page anomaly map — bold red outline = anomalous page" + (f" · {title_label}" if title_label else ""),
                 fontsize=12, fontweight="bold", loc="left")
    cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Anomaly score (Mahalanobis distance)", fontsize=8)
    cbar.ax.axhline(threshold, color=RED, lw=1.3)
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

## 4 · Run the pipeline

Now we use the code above on your documents. Cells run top-to-bottom.

In [ ]:
# Show matplotlib charts inline, and quieten the informational warnings the model loader prints.
%matplotlib inline
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

### Run · Stage 1 — load the model, then PDF → page images

We build the frozen DiT embedder (first run downloads ~330 MB), count the pages in every document,
and render the first page of one genuine document so you can see what the model receives. **Every**
page is processed in Stage 2 — the count below is how many page images that will be.

In [ ]:
embedder = DiTEmbedder(model_id=MODEL, device=DEVICE)
print(f"model   : {embedder.model_id}")
print(f"readout : {embedder.readout}  (mean-pooled patch tokens for DiT)")
print(f"dim     : {embedder.dim}")

ref_files = list_documents(REFERENCE_DIR)
assert len(ref_files) >= 3, "need at least 3 documents in reference/ to build the model"
if UNSUPERVISED:
    data_files = ref_files          # no ground truth: the pile is scored against itself (leave-one-out)
    print("\nUNSUPERVISED mode: no separate reference — every document is scored vs the REST of the set.")
else:
    data_files = list_documents(DATA_DIR)
    assert len(data_files) >= 1, "no documents to classify in data/  (or set UNSUPERVISED = True)"

def page_count(path):
    return len(document_to_pages(path.read_bytes()))

ref_pages  = {p.name: page_count(p) for p in ref_files}
data_pages = ref_pages if UNSUPERVISED else {p.name: page_count(p) for p in data_files}
kind = "documents in the set" if UNSUPERVISED else "reference"
print(f"\n{kind:22}: {len(ref_files)} documents, {sum(ref_pages.values())} pages total")
if not UNSUPERVISED:
    print(f"{'data':22}: {len(data_files)} documents, {sum(data_pages.values())} pages total")

# Render page 1 of the first document, just to show what a page image looks like.
sample = ref_files[0]
sample_img = document_to_pages(sample.read_bytes())[0]
print(f"\nsample page — {sample.name} p1: {sample_img.size[0]}x{sample_img.size[1]} px, mode={sample_img.mode}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.2, 6.8))
ax.imshow(sample_img)
ax.set_title(f"Stage 1 output — one rasterized page\n{sample.name} (page 1 of {ref_pages[sample.name]})", fontsize=10)
ax.set_xlabel("pixels (x)"); ax.set_ylabel("pixels (y)")
plt.show()

### Run · Stage 2 — page image → embedding, then embed every genuine page

`embed()` turns one page image into a 768-number vector. We embed **every page** of every genuine
document into a matrix `X_ref` of shape *(total_genuine_pages, 768)* — this is our definition of a
"normal page". Alongside it we keep `ref_index`, a list of `(filename, page_number, n_pages)` so we
always know which row came from which page.

In [ ]:
sample_vec = np.array(embedder.embed(sample_img), dtype=float)
print(f"one embedding -> shape {sample_vec.shape}, first 8 values {np.round(sample_vec[:8], 4)}")

def embed_corpus(files):
    """Embed EVERY page of every file. Returns (X, index): X is (total_pages, 768);
    index[i] = (filename, page_number, n_pages_in_that_file)."""
    vectors, index = [], []
    for p in files:
        pages = document_to_pages(p.read_bytes())
        for page_no, img in enumerate(pages, start=1):
            vectors.append(embedder.embed(img))
            index.append((p.name, page_no, len(pages)))
    return np.array(vectors, dtype=float), index

print(f"\nembedding all pages of {len(ref_files)} reference documents ...")
X_ref, ref_index = embed_corpus(ref_files)
print(f"reference matrix X_ref: {X_ref.shape}   ({X_ref.shape[0]} genuine pages)")

### Run · Stage 3 — fit the "normal-page" model and see where the threshold comes from

`NormalModel.fit` fits the genuine-page Gaussian and derives the alarm threshold as the **95th
percentile of the genuine leave-one-out scores** (each genuine page scored against all the *other*
genuine pages). The second cell shows the highest genuine-page scores and which pages the threshold
lands between. Using all pages (dozens, not ~15) makes this threshold noticeably more stable than a
one-page-per-document version.

In [ ]:
model = NormalModel.fit(X_ref, threshold_percentile=PERCENTILE)
print(f"threshold : {model.threshold:.2f}   (= {PERCENTILE:.0f}th percentile of genuine per-page leave-one-out scores)")
print(f"shrinkage : {model.shrinkage:.2f}   (0 = raw covariance, 1 = fully shrunk to identity)")
print(f"fitted on : {X_ref.shape[0]} genuine pages")
print(f"genuine per-page leave-one-out score range: {model.ref_scores.min():.2f} .. {model.ref_scores.max():.2f}")

In [ ]:
loo = model.ref_scores
labels = [f"{_stem(name)} p{pg}" for (name, pg, npg) in ref_index]
order = np.argsort(loo)

TOP = 12
print(f"The {TOP} highest-scoring genuine pages (these are what set the threshold):\n")
for r, i in enumerate(order[::-1][:TOP], 1):
    print(f"  {r:2}. {_short_name(labels[i], 22):22} {loo[i]:7.2f}  {'#' * int(loo[i] / 2)}")

n = len(loo); s = np.sort(loo)
pos = (PERCENTILE / 100) * (n - 1); lo, hi = int(np.floor(pos)), int(np.ceil(pos))
print(f"\nnp.percentile(loo, {PERCENTILE:.0f}) with n={n} genuine pages:")
print(f"  index = {PERCENTILE/100:.2f} x ({n}-1) = {pos:.2f}  ->  interpolate sorted[{lo}]={s[lo]:.2f} .. sorted[{hi}]={s[hi]:.2f}")
print(f"  threshold = {s[lo]:.2f} + {pos-lo:.2f} x ({s[hi]:.2f} - {s[lo]:.2f}) = {np.percentile(loo, PERCENTILE):.2f}")
print("\nThreshold at other percentile choices (what changing PERCENTILE would give):")
for p in (90, 95, 99, 100):
    print(f"  percentile {p:3} -> threshold {np.percentile(loo, p):7.2f}")

### Run · Stage 4 — score every page, then decide per document

Every page of every `data/` document is embedded and scored. A **page** is `anomaly` if its score
exceeds the threshold; a **document** is `anomaly` if **any** of its pages is. We report the worst
page and list all flagged pages, so a forged page hidden behind a genuine one is caught and named.

In [ ]:
from collections import OrderedDict

if UNSUPERVISED:
    # Honest anomaly score = leave-one-out: each document scored against the OTHERS. Already computed
    # during the fit (model.ref_scores), so no separate scoring pass is needed and no self-comparison.
    print(f"UNSUPERVISED: using the leave-one-out scores of all {len(ref_index)} pages as anomaly scores.\n")
    eval_index  = ref_index
    page_scores = [float(s) for s in model.ref_scores]
else:
    print(f"embedding + scoring all pages of {len(data_files)} document(s) ...\n")
    X_data, eval_index = embed_corpus(data_files)
    page_scores = [model.score(x) for x in X_data]
page_verdicts = ["anomaly" if sc > model.threshold else "normal" for sc in page_scores]

# Group page results back into documents (a document is anomaly if ANY of its pages is).
docs = OrderedDict()
for (name, pg, npg), sc, v in zip(eval_index, page_scores, page_verdicts):
    docs.setdefault(name, []).append({"page": pg, "n_pages": npg, "score": sc, "verdict": v})

doc_rows = []
for name, pages in docs.items():
    flagged = [p["page"] for p in pages if p["verdict"] == "anomaly"]
    worst = max(pages, key=lambda p: p["score"])
    doc_rows.append({
        "name": name, "n_pages": pages[0]["n_pages"],
        "n_flagged": len(flagged), "flagged_pages": flagged,
        "worst_page": worst["page"], "worst_score": worst["score"],
        "verdict": "anomaly" if flagged else "normal",
    })

n_doc_anom  = sum(1 for d in doc_rows if d["verdict"] == "anomaly")
n_page_anom = page_verdicts.count("anomaly")
print(f"DOCUMENTS: {n_doc_anom} anomaly, {len(doc_rows) - n_doc_anom} normal   (threshold = {model.threshold:.2f})")
print(f"PAGES    : {n_page_anom} anomaly, {len(page_scores) - n_page_anom} normal\n")

# List the most unusual documents (top of the distribution) — the ones to inspect first.
TOPN = min(20, len(doc_rows))
print(f"Most unusual {TOPN} documents (highest score first):")
print(f"  {'document':30} {'score':>8}   verdict")
print("  " + "-" * 52)
for d in sorted(doc_rows, key=lambda d: -d["worst_score"])[:TOPN]:
    flag = "  <== ANOMALY" if d["verdict"] == "anomaly" else ""
    print(f"  {_short_name(d['name'], 30):30} {d['worst_score']:8.2f}   {d['verdict']:8}{flag}")

In [ ]:
import csv
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1) Per-PAGE results — one row per page, so nothing is hidden by aggregation.
page_csv = RESULT_DIR / f"{ts}_page_results.csv"
with page_csv.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["filename", "page", "n_pages", "mahalanobis_score", "verdict"])
    for (name, pg, npg), sc, v in zip(eval_index, page_scores, page_verdicts):
        w.writerow([name, pg, npg, f"{sc:.2f}", v])

# 2) Per-DOCUMENT summary — sorted most-unusual first, so anomalies are at the top.
doc_csv = RESULT_DIR / f"{ts}_document_summary.csv"
with doc_csv.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["filename", "n_pages", "n_flagged_pages", "flagged_pages", "worst_page", "max_score", "verdict"])
    for d in sorted(doc_rows, key=lambda d: -d["worst_score"]):
        w.writerow([d["name"], d["n_pages"], d["n_flagged"],
                    " ".join(f"p{p}" for p in d["flagged_pages"]),
                    f"p{d['worst_page']}", f"{d['worst_score']:.2f}", d["verdict"]])

print("saved:", page_csv)
print("saved:", doc_csv, " (sorted most-unusual first)")

In [ ]:
import tempfile
from IPython.display import Image as IPyImage, display

with tempfile.TemporaryDirectory() as td:
    dist_png = Path(td) / "dist.png"
    plot_page_distribution(model, eval_index, page_scores, page_verdicts, dist_png, DOC_TYPE,
                           unsupervised=UNSUPERVISED)
    display(IPyImage(filename=str(dist_png)))
    if UNSUPERVISED:
        # the "distribution" view — a histogram of scores with the threshold marked
        hist_png = Path(td) / "hist.png"
        plot_score_histogram(page_scores, model.threshold, hist_png, DOC_TYPE)
        display(IPyImage(filename=str(hist_png)))
    else:
        # the per-page fraud-localisation grid (rows = documents, columns = pages)
        map_png = Path(td) / "map.png"
        plot_page_map(eval_index, page_scores, page_verdicts, model.threshold, map_png, DOC_TYPE)
        display(IPyImage(filename=str(map_png)))

## 5 · How to read the result, and how to tune it

**Everything is per page.** The `data/` verdict table is per document (anomaly if *any* page is), but
the two CSVs give you both levels: `..._page_results.csv` has one row per page, and
`..._document_summary.csv` has the headline verdict plus exactly which pages tripped it.

**Reading the charts**
- *Per-page anomaly scores:* one dot per page. Teal dots low on the axis are genuine reference pages;
  the dashed line is the threshold; data pages past it (red) are flagged, labelled `file p<N>`.
- *Per-page anomaly map:* a grid of documents (rows) × pages (columns), each cell coloured by its
  score. A **bold red outline** marks an anomalous page and its score is printed in the cell — this is
  the "which page is fake" view. An empty spot means the document has no such page number.

**Tuning**
- **Sensitivity:** raise `PERCENTILE` (e.g. 99) for fewer alarms, lower it for more. It only moves
  borderline pages; pages scoring far above threshold stay flagged.
- **Resolution:** `PDF_RENDER_DPI` (Stage 3.1) controls how sharply pages are rasterized. 200 is a good
  default; raising it captures finer print at the cost of speed and memory.
- **Another document type:** create `reference/<type>/` and `data/<type>/`, set `DOC_TYPE`, and re-run.

**Things to check before trusting a verdict**
- Every page in `reference/` must be genuine — the model treats *all* genuine pages as the definition
  of "normal", so one bad reference page widens what counts as normal.
- **Heterogeneous pages cause false positives.** A statement's pages differ (a cover/summary page vs
  transaction pages vs a terms-and-conditions page). If a page *type* appears in `data/` but never in
  any genuine reference document, it can be flagged for being unusual rather than fake — so treat a
  single flagged page as "look here", not proof. Adding genuine examples that include every page type
  you expect reduces this.
- A whole document scoring high on *every* page usually means it's a different kind of document (or a
  different layout family) than your references — not necessarily fraud.

## 6 · Unsupervised mode (no ground truth)

Set `UNSUPERVISED = True` when you have **no known-genuine reference set** — just a pile of documents
you want to screen. Put them all in `reference/<type>/`, leave `data/<type>/` empty, and Run-All.

**What changes:** every document is scored by its **leave-one-out** distance — how far it sits from the
cloud formed by *all the other* documents — instead of being compared to a separate reference. The
alarm threshold is still the `PERCENTILE` (95th) of that distribution, so the **most unusual ~5%** are
flagged automatically. Nothing is compared to itself, and you never set a threshold by hand.

**What you get:**
- the **anomaly-score distribution** (a strip plot + a histogram) with the threshold line — the shape
  of your data, and where the unusual tail begins;
- a **"most unusual documents"** list (console) and `..._document_summary.csv` **sorted worst-first** —
  the cards to inspect are at the top.

**How to read it:** treat a flagged document as *"looks different from the rest — look here"*, not as
proof. If most of the pile is genuine, the top of the distribution is where odd scans, wrong-type
images, or tampering tend to surface. Because the unusual documents are *inside* the set, they slightly
widen "normal"; for a sharper pass you can flag the top few percent, remove them, and re-run.